# Hong Kong GFA Concession Exploration 

In [15]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from tabula.io import read_pdf
import requests
import pdfplumber
from io import BytesIO
import tempfile
import re

### Data Cleaning

In [16]:
df = pd.read_csv("../data/BDGFA_20260510_gdb_BDGFA_converted.csv", header=0) 
df.head()

,OBJECTID,DATASET_EN,DATASET_TC,ADDRESS_EN,ADDRESS_TC,SEARCH01_EN,SEARCH01_TC,SEARCH02_EN,SEARCH02_TC,NSEARCH01_EN,...,NSEARCH09_TC,NSEARCH10_EN,NSEARCH10_TC,NORTHING,EASTING,LATITUDE,LONGITUDE,LASTUPDATE,GeometryEasting,GeometryNorthing
0,1,Summary of Gross Floor Area (GFA) Concessions ...,私人發展項目總樓面面積寬免概況及相關資料,"15 Sheung Yuet Road, Kowloon 2.8.0/(26)",九龍 常悅道 15 號 2.8.0/(26),2025,2025,KOWLOON & NEW TERRITORIES,九龍及新界,KN8/2025/OP,...,不適用,https://www.bd.gov.hk/doc/en/resources/codes-a...,https://www.bd.gov.hk/doc/en/resources/codes-a...,820347.800000,839671.600000,22.322140,114.20991,2026-04-23T10:11:22,839671.6000,820347.8000
1,2,Summary of Gross Floor Area (GFA) Concessions ...,私人發展項目總樓面面積寬免概況及相關資料,"101 King's Road, Hong Kong. 1.5.1/(11)","101 King's Road, Hong Kong. 1.5.1/(11)",2025,2025,HONG KONG ISLAND,香港,HK53/2025/OP,...,https://www.bd.gov.hk/doc/en/resources/codes-a...,https://www.bd.gov.hk/doc/en/resources/codes-a...,https://www.bd.gov.hk/doc/en/resources/codes-a...,816469.600000,837884.500000,22.287120,114.19256,2026-04-23T10:10:57,837884.5000,816469.6000
2,3,Summary of Gross Floor Area (GFA) Concessions ...,私人發展項目總樓面面積寬免概況及相關資料,"18 Ap Lei Chau Praya Road, Hong Kong 1.7.4/(9)...","18 Ap Lei Chau Praya Road, Hong Kong 1.7.4/(9)...",2025,2025,HONG KONG ISLAND,香港,HK49/2025/OP,...,https://www.bd.gov.hk/doc/en/resources/codes-a...,https://www.bd.gov.hk/doc/en/resources/codes-a...,https://www.bd.gov.hk/doc/en/resources/codes-a...,811385.008142,834384.155388,22.241203,114.15860,2026-04-23T10:10:41,834384.1554,811385.0081
3,4,Summary of Gross Floor Area (GFA) Concessions ...,私人發展項目總樓面面積寬免概況及相關資料,"281 Gloucester Road, Hong Kong 1.4.6/(3)","281 Gloucester Road, Hong Kong 1.4.6/(3)",2025,2025,HONG KONG ISLAND,香港,HK37/2025/OP,...,不適用,Not applicable,不適用,815883.700000,837043.600000,22.281830,114.18440,2026-04-23T10:08:59,837043.6000,815883.7000
4,5,Summary of Gross Floor Area (GFA) Concessions ...,私人發展項目總樓面面積寬免概況及相關資料,"9 Eastern Street, Hong Kong 1.1.2/(20)","9 Eastern Street, Hong Kong 1.1.2/(20)",2025,2025,HONG KONG ISLAND,香港,HK34/2025/OP,...,https://www.bd.gov.hk/doc/en/resources/codes-a...,https://www.bd.gov.hk/doc/en/resources/codes-a...,https://www.bd.gov.hk/doc/en/resources/codes-a...,816511.900000,832840.000000,22.287500,114.14361,2026-04-23T10:08:31,832840.0000,816511.9000


In [17]:
df.shape

(651, 36)

In [18]:
## Remove irrelevant columns and keep everything in English

for column in df.columns:
    if "_TC" in column:
        df.drop(columns={column}, inplace=True)

df.columns

Index(['OBJECTID', 'DATASET_EN', 'ADDRESS_EN', 'SEARCH01_EN', 'SEARCH02_EN',
       'NSEARCH01_EN', 'NSEARCH02_EN', 'NSEARCH03_EN', 'NSEARCH04_EN',
       'NSEARCH05_EN', 'NSEARCH06_EN', 'NSEARCH07_EN', 'NSEARCH08_EN',
       'NSEARCH09_EN', 'NSEARCH10_EN', 'NORTHING', 'EASTING', 'LATITUDE',
       'LONGITUDE', 'LASTUPDATE', 'GeometryEasting', 'GeometryNorthing'],
      dtype='str')

| Field Name | English Description |
|---|---|
| DATASET_EN | Dataset |
| ADDRESS_EN | Address |
| SEARCH01_EN | Year |
| SEARCH02_EN | District |
| NSEARCH01_EN | Permit No. |
| NSEARCH02_EN | Date of Occupation Permit |
| NSEARCH03_EN | Building Type |
| NSEARCH04_EN | Use of Building |
| NSEARCH05_EN | Summary of GFA Concessions |
| NSEARCH06_EN | Greenery Area |
| NSEARCH07_EN | Estimated Energy Performance / consumption |
| NSEARCH08_EN | BEAM Plus Assessment Result |
| NSEARCH09_EN | Residential Thermal Transfer Value |
| NSEARCH10_EN | Overall Thermal Transfer Value of Residents' Recreational Facilities |

#### Using Python to Access PDF Tables

In [19]:
## Apply onto Summary of GFA Concessions First

url = df[df['OBJECTID'] == 1]['NSEARCH05_EN'].values ## returns a list object

r = requests.get(url[0])

In [20]:
with pdfplumber.open(BytesIO(r.content)) as pdf:
    page = pdf.pages[0]
    W = page.width 
    H = page.height
    if W < H:
        page = page.rotate(-page.rotation)

    ## Crop the page 
    cropped = page.crop(
        (0.38*W, 
         0.24*H, 
         0.55*W, 
         0.78*H)
    )
    
    table = cropped.extract_table()

table

[['', '環保設施等(平方米)', None, None, None],
 [None, 'Domestic\n住用部分', None, 'Non-domestic\n非住用部分', None],
 ['', '5', '', '7', ''],
 ['', '6', '', '8', '1041.88'],
 ['', '7', '', '9', ''],
 ['', '8', '', '10', ''],
 ['',
  '9\n10\n11\n12\n13\n14\n15\n16\n17A4\n17B4\n18\n19\n20\n21\n22\n23\n26\n27\n28\n29\n30\n31\n32\n33',
  '',
  '11\n13\n14\n17A4\n17B4\n18\n19\n20\n21\n22\n23\n24\n25\n27\n28\n29\n30\n31\n32\n33\n34\n35\n36\n38',
  '349.048\n237.024\n347.315\n505.986'],
 ['', '34\n35\n36\n38', '', '', ''],
 [None, '', '0', '', None],
 [None, '', None, '', None]]

In [21]:
# with tempfile.NamedTemporaryFile(suffix="pdf") as tmp:
#     tmp.write(r.content)
#     tmp.flush()

#     dfs = read_pdf(
#         tmp.name,
#         pages=1,
#         multiple_tables=True
#     )

# gfa_summary = dfs[0]
# gfa_summary

In [70]:
df[(~df['max_item'].isin([True, False]))]['SEARCH01_EN'].value_counts(sort=True)

SEARCH01_EN
2018    140
2019    125
2020    105
2021     61
2025      3
2022      1
Name: count, dtype: int64

In [71]:
df_f = df[df['SEARCH01_EN'] >= 2019].copy()
df_f.shape

(498, 23)

In [73]:
df_f[(df_f['max_item'].isin([True, False]))]['SEARCH01_EN'].value_counts(sort=True)

SEARCH01_EN
2022    93
2023    38
2021    29
2025    18
2019    12
2024     7
2020     6
Name: count, dtype: int64